Walking Running Classification

Project Objective

The goal is to build a machine learning model that can classify whether a person is Walking or Running using sensor readings from wearable devices.

import Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

Load Dataset

In [ ]:
df = pd.read_csv("walkrun.csv")

df.head()

Data Understanding

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

Data Cleaning
#Check missing values

In [ ]:
df.isnull().sum()

Check duplicates

In [ ]:
df.duplicated().sum()

Check Outliers

Box Plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.boxplot(data=df[['acceleration_x']])

plt.show()

In [ ]:
df.boxplot(figsize=(12,6))
plt.xticks(rotation=45)
plt.show()

In [ ]:
Q1 = df['acceleration_x'].quantile(0.25)

Q3 = df['acceleration_x'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR

upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['acceleration_x'] < lower_bound) |
    (df['acceleration_x'] > upper_bound)
]

print("Number of Outliers:", len(outliers))

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(col, ":", len(outliers))

Feature Engineering

Date, Time, and Username may not help prediction.

In [ ]:
df.drop(
    ['date', 'time', 'username'],
    axis=1,
    inplace=True
)

Encode target:

In [ ]:
le = LabelEncoder()

df['activity'] = le.fit_transform(df['activity'])

EDA

In [ ]:
sns.countplot(
    x='activity',
    data=df
)

plt.show()

Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    df.corr(),
    annot=True
)

plt.show()

Distribution Plot

In [ ]:
df.hist(
    figsize=(12,10)
)

plt.show()

Feature and Target Separation

In [ ]:
X = df.drop(
    'activity',
    axis=1)
y = df['activity']

Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

Model Building

In [ ]:
lr = LogisticRegression()
dt = DecisionTreeClassifier()
rf = RandomForestClassifier()
knn = KNeighborsClassifier()

lr.fit(X_train,y_train)
dt.fit(X_train,y_train)
rf.fit(X_train,y_train)
knn.fit(X_train,y_train)

Model Comparison

In [ ]:
models = {
    "Logistic Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "KNN": knn
}

results = []

for name, model in models.items():

    pred = model.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test,pred)
    ])

comparison = pd.DataFrame(
    results,
    columns=['Model','Accuracy']
)

comparison

Confusion Matrix

In [ ]:
sns.heatmap(
    confusion_matrix(y_test, rf.predict(X_test)),
    annot=True,
    fmt='d'
)

plt.show()

ROC Curve

In [ ]:
rf_prob = rf.predict_proba(X_test)[:,1]

fpr,tpr,_ = roc_curve(
    y_test,
    rf_prob
)

auc = roc_auc_score(
    y_test,
    rf_prob
)

print(auc)

In [ ]:
plt.plot(
    fpr,
    tpr,
    label=f"AUC={auc:.2f}"
)

plt.plot([0,1],[0,1],'--')

plt.legend()

plt.show()

Cross Validation

In [ ]:
cv_scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print(cv_scores.mean())

Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators':[50,100,200],
    'max_depth':[5,10,None]
}

grid = GridSearchCV(
    RandomForestClassifier(),
    params,
    cv=5
)

grid.fit(
    X_train,
    y_train
)

print(
    grid.best_params_
)

Feature Importance

In [ ]:
importance = pd.DataFrame({
    'Feature':X.columns,
    'Importance':rf.feature_importances_
})

importance.sort_values(
    by='Importance',
    ascending=False,
    inplace=True
)

In [ ]:
sns.barplot(
    x='Importance',
    y='Feature',
    data=importance
)

plt.show()

In [ ]:
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.model_selection import cross_val_score

models = {
    'Logistic Regression': lr,
    'Decision Tree': dt,
    'Random Forest': rf,
    'KNN': knn
}

results = []

for name, model in models.items():

    # Predictions
    y_pred = model.predict(X_test)

    # Probability predictions for ROC-AUC
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    roc_auc = roc_auc_score(y_test, y_prob)

    cv_accuracy = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='accuracy'
    ).mean()

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        roc_auc,
        cv_accuracy
    ])

comparison_df = pd.DataFrame(
    results,
    columns=[
        'Model',
        'Accuracy',
        'Precision',
        'Recall',
        'F1-Score',
        'ROC-AUC Score',
        'Cross Validation Accuracy'
    ]
)

comparison_df = comparison_df.sort_values(
    by='Accuracy',
    ascending=False
)

print(comparison_df)

In [ ]:
comparison_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}',
    'ROC-AUC Score': '{:.4f}',
    'Cross Validation Accuracy': '{:.4f}'
})

<b>Challenges Faced</b></br>
Sensor data contains many numerical readings.</br>
Features were on different scales.</br>
Selecting relevant features.</br>
Comparing multiple classification algorithms.</br>
Avoiding overfitting.</br>
<b>Solution</b></br>
Applied feature scaling.</br>
Used correlation analysis.</br>
Performed cross-validation.</br>
Used hyperparameter tuning.</br>
Compared multiple models.</br>

<b>Project Risks</b></br>
Sensor noise may affect predictions.</br>
Real-world movement patterns may differ.</br>
Overfitting on training data.</br>
Performance may degrade with new users.</br>

Among all the machine learning models evaluated, <b>Random Forest</b> achieved the best overall performance based on Accuracy, Precision, Recall, F1-Score, ROC-AUC Score, and Cross Validation Accuracy. Therefore, Random Forest was selected as the <b>final model</b> for activity classification.